In [6]:

import requests
import urllib.parse
import pandas as pd

API_URL = "http://localhost:8080/api"

def get_all_results(url): 
    results = []
    page = 0
    while True:
        response = requests.get(url, params={"page": page, "size": 1000})
        if response.status_code != 200:
            raise Exception(f"Failed to retrieve data: {response.status_code}")
        data = response.json()
        elements = data.get("elements", [])
        results.extend(elements)
        page += 1
        if page > data.get("totalPages", 0):
            break
    return results

def encode_iri(iri):
    return urllib.parse.quote(urllib.parse.quote(iri, safe=''), safe='')


def to_df(classes):
    rows = []
    for cls in classes:
        row = {
            "iri": cls.get("iri"),
            "curie": cls.get("curie"),
            "label": cls.get("label")[0]
        }
        rows.append(row)
    return pd.DataFrame(rows)

all_hp = get_all_results(f'{API_URL}/v2/ontologies/hp/classes?isDefiningOntology=true')
all_mp = get_all_results(f'{API_URL}/v2/ontologies/mp/classes?isDefiningOntology=true')
all_zp = get_all_results(f'{API_URL}/v2/ontologies/zp/classes?isDefiningOntology=true')

to_df(all_hp).to_csv("hp.csv", index=False)
to_df(all_mp).to_csv("mp.csv", index=False)
to_df(all_zp).to_csv("zp.csv", index=False)


In [ ]:

def get_embedding_vector(ontology_id, iri):
    url = f"{API_URL}/v2/ontologies/{ontology_id}/classes/{encode_iri(iri)}/embeddingVector"
    response = requests.get(url)
    if response.status_code != 200:
        raise Exception(f"Failed to retrieve embedding vector: {response.status_code}")
    return response.text

hp = pd.read_csv("hp.csv")
mp = pd.read_csv("mp.csv")
zp = pd.read_csv("zp.csv")

for index, row in hp.iterrows():
    hp.at[index, "embedding"] = get_embedding_vector("hp", row["iri"])
for index, row in mp.iterrows():
    mp.at[index, "embedding"] = get_embedding_vector("mp", row["iri"])
for index, row in zp.iterrows():
    zp.at[index, "embedding"] = get_embedding_vector("zp", row["iri"])

hp.to_csv("hp_with_embeddings.csv.gz", index=False, compression="gzip")
mp.to_csv("mp_with_embeddings.csv.gz", index=False, compression="gzip")
zp.to_csv("zp_with_embeddings.csv.gz", index=False, compression="gzip")


In [8]:

def get_embedding_vector(ontology_id, iri):
    url = f"{API_URL}/v2/ontologies/{ontology_id}/classes/{encode_iri(iri)}/embeddingVector"
    response = requests.get(url)
    if response.status_code != 200:
        raise Exception(f"Failed to retrieve embedding vector: {response.status_code}")
    return response.text

zp = pd.read_csv("zp.csv")

for index, row in zp.iterrows():
    zp.at[index, "embedding"] = get_embedding_vector("zp", row["iri"])

zp.to_csv("zp_with_embeddings.csv.gz", index=False, compression="gzip")

In [4]:
import json
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from multiprocessing import Pool, cpu_count, Manager, Lock

hp = pd.read_csv("hp_with_embeddings.csv.gz", compression="gzip")
mp = pd.read_csv("mp_with_embeddings.csv.gz", compression="gzip")

hp["embedding"] = hp["embedding"].apply(json.loads)
mp["embedding"] = mp["embedding"].apply(json.loads)

mp_embeddings = np.array(mp["embedding"].tolist())
mp_metadata = mp[["curie", "label"]].values.tolist()

def compute_similarities(args):
    hp_row, counter, total, lock = args
    index, row = hp_row
    hp_embedding = np.array(row["embedding"]).reshape(1, -1)
    sims = cosine_similarity(hp_embedding, mp_embeddings)[0]

    results = []
    for i, sim in enumerate(sims):
        if sim >= 0.3:
            results.append({
                "hp_curie": row["curie"],
                "hp_label": row["label"],
                "mp_curie": mp_metadata[i][0],
                "mp_label": mp_metadata[i][1],
                "similarity": sim
            })

    with lock:
        counter.value += 1
        if counter.value % 1000 == 0 or counter.value == total:
            print(f"- {counter.value} of {total} terms processed...")

    return results

if __name__ == "__main__":
    manager = Manager()
    counter = manager.Value('i', 0)
    lock = manager.Lock()

    total = len(hp)
    args = [(row, counter, total, lock) for row in hp.iterrows()]

    with Pool(processes=cpu_count()) as pool:
        results = pool.map(compute_similarities, args)

    similarities = [item for sublist in results for item in sublist]
    similarities_df = pd.DataFrame(similarities)
    similarities_df.to_csv("hp_mp.csv.gz", index=False, compression="gzip")


- 1000 of 19177 terms processed...
- 2000 of 19177 terms processed...
- 3000 of 19177 terms processed...
- 4000 of 19177 terms processed...
- 5000 of 19177 terms processed...
- 6000 of 19177 terms processed...
- 7000 of 19177 terms processed...
- 8000 of 19177 terms processed...
- 9000 of 19177 terms processed...
- 10000 of 19177 terms processed...
- 11000 of 19177 terms processed...
- 12000 of 19177 terms processed...
- 13000 of 19177 terms processed...
- 14000 of 19177 terms processed...
- 15000 of 19177 terms processed...
- 16000 of 19177 terms processed...
- 17000 of 19177 terms processed...
- 18000 of 19177 terms processed...
- 19000 of 19177 terms processed...
- 19177 of 19177 terms processed...
